# Chest X-ray pneumonia classifier — original dataset, no API token

This notebook uses the same Chest X-Ray Images (Pneumonia) dataset as your original notebook, trains a corrected model, validates it correctly, and downloads the files required by the Streamlit website. It is for research and education only, not diagnosis or treatment.

**Before running:** select **Runtime → Change runtime type → T4 GPU** in Colab. No Kaggle API token is used by this notebook.

In [ ]:
# Check that Colab has a GPU. The notebook can run on CPU, but it is much slower.
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('Available GPU(s):', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('WARNING: No GPU found. Select Runtime > Change runtime type > T4 GPU, then reconnect and rerun.')

In [ ]:
# Install the same public-dataset downloader used in your original notebook.
!pip -q install kagglehub
import kagglehub
from pathlib import Path

In [ ]:
# This is the same public dataset identifier used in your original notebook. No token is requested.
download_path = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
roots = [p for p in [download_path, *download_path.rglob('chest_xray')] if (p / 'train').is_dir() and (p / 'test').is_dir()]
if not roots:
    raise FileNotFoundError(f'Could not find train/ and test/ folders under {download_path}.')
DATASET_ROOT = roots[0]
print('Using dataset:', DATASET_ROOT)

In [ ]:
# Make a stratified validation set from TRAIN only. The official test set stays untouched.
import json, random, shutil
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']  # model output index 0, then 1
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

def list_images(split):
    paths, labels = [], []
    for index, class_name in enumerate(CLASS_NAMES):
        files = [p for p in (DATASET_ROOT / split / class_name).rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
        paths.extend(map(str, files)); labels.extend([index] * len(files))
    return np.array(paths), np.array(labels, dtype=np.int32)

all_train_paths, all_train_labels = list_images('train')
test_paths, test_labels = list_images('test')
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_train_paths, all_train_labels, test_size=0.10, random_state=SEED, stratify=all_train_labels)

def make_unbatched_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    def read_image(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        return tf.cast(image, tf.float32), tf.cast(label, tf.float32)
    return dataset.map(read_image, num_parallel_calls=tf.data.AUTOTUNE)

# Balance TRAINING only: retain every pneumonia image, repeatedly sample Normal images,
# and let the model's augmentation create varied Normal examples. Validation and test
# retain their natural distributions so their metrics remain honest.
normal_paths = train_paths[train_labels == 0]
pneumonia_paths = train_paths[train_labels == 1]
normal_stream = make_unbatched_dataset(normal_paths, np.zeros(len(normal_paths), dtype=np.int32), training=True).repeat()
pneumonia_stream = make_unbatched_dataset(pneumonia_paths, np.ones(len(pneumonia_paths), dtype=np.int32), training=True).repeat()
train_ds = tf.data.Dataset.sample_from_datasets([normal_stream, pneumonia_stream], weights=[0.5, 0.5], seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
# One epoch contains each pneumonia training image once on average, with an equal number of Normal examples.
STEPS_PER_EPOCH = int(np.ceil((2 * len(pneumonia_paths)) / BATCH_SIZE))
val_ds = make_unbatched_dataset(val_paths, val_labels).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = make_unbatched_dataset(test_paths, test_labels).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
training_counts = {name: int((train_labels == i).sum()) for i, name in enumerate(CLASS_NAMES)}
print('Train:', len(train_paths), training_counts)
print('Validation:', len(val_paths), {name: int((val_labels == i).sum()) for i, name in enumerate(CLASS_NAMES)})
print('Untouched test:', len(test_paths), {name: int((test_labels == i).sum()) for i, name in enumerate(CLASS_NAMES)})
print('Balanced training stream: 50% NORMAL / 50% PNEUMONIA')
print('Steps per epoch:', STEPS_PER_EPOCH)

In [ ]:
# Build the model. Preprocessing is inside the saved model: website inputs must be RGB pixels in range 0–255.
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='xray')
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomTranslation(0.03, 0.03),
    tf.keras.layers.RandomContrast(0.08)], name='augmentation')
x = augmentation(x)
backbone = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
backbone.trainable = False
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='pneumonia_probability')(x)
model = tf.keras.Model(inputs, outputs, name='pneumonia_mobilenet_v2')

def compile_model(learning_rate):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss=tf.keras.losses.BinaryCrossentropy(), metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

compile_model(1e-3)
model.summary()

In [ ]:
# Train the head, then fine-tune only the final 30 MobileNet layers.
ARTIFACTS = Path('/content/artifacts')
ARTIFACTS.mkdir(exist_ok=True)
MODEL_PATH = ARTIFACTS / 'best_model.keras'
head_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)]
history_head = model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, validation_data=val_ds, epochs=12, callbacks=head_callbacks)

model = tf.keras.models.load_model(MODEL_PATH)
backbone = next(layer for layer in model.layers if isinstance(layer, tf.keras.Model) and layer.name.lower().startswith('mobilenetv2'))
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
compile_model(1e-5)
fine_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True)]
history_fine = model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, validation_data=val_ds, epochs=6, callbacks=fine_callbacks)
model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
# Select the decision threshold using validation only, then evaluate once on the untouched test data.
import matplotlib.pyplot as plt
import seaborn as sns
val_probabilities = model.predict(val_ds, verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 181)
balanced_accuracies = []
for threshold_candidate in thresholds:
    predicted = val_probabilities >= threshold_candidate
    sensitivity = predicted[val_labels == 1].mean()
    specificity = (~predicted[val_labels == 0]).mean()
    balanced_accuracies.append((sensitivity + specificity) / 2)
threshold = float(thresholds[int(np.argmax(balanced_accuracies))])

test_probabilities = model.predict(test_ds, verbose=0).ravel()
test_predictions = test_probabilities >= threshold
tp = int(((test_predictions == 1) & (test_labels == 1)).sum())
tn = int(((test_predictions == 0) & (test_labels == 0)).sum())
fp = int(((test_predictions == 1) & (test_labels == 0)).sum())
fn = int(((test_predictions == 0) & (test_labels == 1)).sum())
test_metrics = {'accuracy': (tp + tn) / len(test_labels), 'sensitivity': tp / max(1, tp + fn), 'specificity': tn / max(1, tn + fp), 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
metadata = {'class_names': ['NORMAL', 'PNEUMONIA'], 'positive_class': 'PNEUMONIA', 'image_size': list(IMAGE_SIZE), 'input_pixels': 'RGB float32 in range 0-255; do not divide by 255', 'threshold': threshold, 'validation_balanced_accuracy': float(max(balanced_accuracies)), 'test_metrics': test_metrics, 'training_class_counts': training_counts, 'seed': SEED}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

# Graphic confusion matrix: each row is a real class and sums to 100%.
cm = np.array([[tn, fp], [fn, tp]])
cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100
labels = np.array([[f'{count}\n({percent:.1f}%)' for count, percent in zip(count_row, percent_row)] for count_row, percent_row in zip(cm, cm_percent)])
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(cm_percent, annot=labels, fmt='', cmap='Blues', vmin=0, vmax=100, square=True, linewidths=1, linecolor='white', cbar_kws={'label': 'Percentage of actual class'}, ax=ax)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_yticklabels(CLASS_NAMES, fontsize=12, rotation=0)
ax.set_xlabel('Predicted label', fontsize=12, labelpad=12)
ax.set_ylabel('Actual label', fontsize=12, labelpad=12)
ax.set_title('Test-set confusion matrix\nCount and percentage within each actual class', fontsize=14, pad=14)
plt.tight_layout()
plt.show()
print(f'Normal correctly identified (specificity): {test_metrics["specificity"]:.1%}')
print(f'Pneumonia correctly identified (sensitivity): {test_metrics["sensitivity"]:.1%}')

In [ ]:
# Download one ZIP file. Extract it, then copy both files into the website's artifacts/ folder.
from google.colab import files
shutil.make_archive('/content/pneumonia_website_artifacts', 'zip', ARTIFACTS)
files.download('/content/pneumonia_website_artifacts.zip')